In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

cwd = Path.cwd()
project_root = cwd
for _ in range(6):
    if (project_root / "src").exists():
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

proj_path = str(project_root.resolve())
if proj_path not in sys.path:
    sys.path.insert(0, proj_path)


import json
import random
import sys
from pathlib import Path

import aerosandbox as asb
import numpy as np
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm

from aerosandbox.geometry.airfoil.airfoil_families import get_kulfan_parameters
from src.airfoil.compute_airfoil_quality import QualityError, compute_airfoil_quality

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

/home/matsouto/Documents/py/AeroGen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extraction and Geometry Filtering

In [2]:
airfoil_database_path = asb._asb_root / "geometry" / "airfoil" / "airfoil_database"
raw_airfoil_database = [
    asb.Airfoil(name=filename.stem).normalize()
    for filename in airfoil_database_path.glob("*.dat")
]

quality_airfoil_database = []
failed_quality = []
for airfoil in tqdm(raw_airfoil_database, desc="Quality check"):
    try:
        compute_airfoil_quality(airfoil, airfoil_database_path)
        quality_airfoil_database.append(airfoil)
    except QualityError as exc:
        failed_quality.append((airfoil.name, str(exc)))

print(f"Raw airfoils:     {len(raw_airfoil_database)}")
print(f"Approved airfoils: {len(quality_airfoil_database)}")
print(f"Rejected airfoils: {len(failed_quality)}")
pd.DataFrame(failed_quality, columns=["airfoil_name", "reason"]).head()

Quality check: 100%|██████████| 2174/2174 [00:07<00:00, 296.48it/s]

Raw airfoils:     2174
Approved airfoils: 2170
Rejected airfoils: 4


,airfoil_name,reason
0,fx79w660a,Airfoil has abnormally large changes in angle ...
1,mh112,Airfoil has abnormally high x-coordinates.
2,as6095,Airfoil has negative thickness.
3,fx79w470a,Airfoil has abnormally large changes in angle ...


## Base Dataset Assembly

In [3]:
N_WEIGHTS_PER_SIDE = 8
N_POINTS_PER_SIDE = 75
LW_CAP = 8.0 # Basically disabled
UW_CAP = 8.0 # Basically disabled
THICKNESS_CAP = 10.0 # Basically disabled
LE_CAP = 10.0 # Basically disabled

def build_airfoil_records(airfoil_records, n_weights_per_side=N_WEIGHTS_PER_SIDE, n_points_per_side=N_POINTS_PER_SIDE):
    rows = []
    rejected = []

    for record in tqdm(airfoil_records, desc="Building airfoil records"):
        airfoil = record["airfoil"]
        try:
            standardized_airfoil = airfoil.normalize().repanel(n_points_per_side)
            parameters = get_kulfan_parameters(
                coordinates=standardized_airfoil.coordinates,
                n_weights_per_side=n_weights_per_side,
                normalize_coordinates=True,
                use_leading_edge_modification=True,
            )
        except Exception as exc:
            rejected.append((airfoil.name, f"kulfan_failed: {exc}"))
            continue

        lower_weights = np.asarray(parameters["lower_weights"], dtype=float)
        upper_weights = np.asarray(parameters["upper_weights"], dtype=float)
        te_thickness = float(parameters["TE_thickness"])
        leading_edge_weight = float(parameters["leading_edge_weight"])

        if (
            np.any(lower_weights > LW_CAP)
            or np.any(upper_weights > UW_CAP)
            or te_thickness > THICKNESS_CAP
            or leading_edge_weight < -LE_CAP
            or leading_edge_weight > LE_CAP
        ):
            rejected.append((airfoil.name, "outlier_filtered"))
            continue

        coords = standardized_airfoil.coordinates
        thickness_proxy = float(np.max(coords[:, 1]) - np.min(coords[:, 1]))

        row = {
            "airfoil_name": airfoil.name,
            "coordinates": coords,
            "lower_weights": lower_weights.tolist(),
            "upper_weights": upper_weights.tolist(),
            "TE_thickness": te_thickness,
            "leading_edge_weight": leading_edge_weight,
            "shape": coords.shape,
            "points": int(coords.shape[0]),
            "thickness_proxy": thickness_proxy,
        }
        row.update({key: value for key, value in record.items() if key != "airfoil"})
        rows.append(row)

    return pd.DataFrame(rows), pd.DataFrame(rejected, columns=["airfoil_name", "reason"])

original_records = [
    {
        "airfoil": airfoil,
        "source_airfoil_name": airfoil.name,
        "augmentation_tag": "original",
        "is_augmented": False,
    }
    for airfoil in quality_airfoil_database
]

base_airfoils_dataset, build_rejections = build_airfoil_records(original_records)
print(f"Base airfoils: {len(base_airfoils_dataset)}")
display(base_airfoils_dataset.head())
display(build_rejections.head())

Building airfoil records: 100%|██████████| 2170/2170 [00:01<00:00, 2110.74it/s]

Base airfoils: 2170


,airfoil_name,coordinates,lower_weights,upper_weights,TE_thickness,leading_edge_weight,shape,points,thickness_proxy,source_airfoil_name,augmentation_tag,is_augmented
0,marske4,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,-1.293263e-01,"(149, 2)",149,0.111491,marske4,original,False
1,HL73-650rev,"[[1.0, 0.0], [0.9995497587046329, 0.0001078248...","[-0.08370574371775676, -0.11286897235829314, 0...","[0.13099602423052709, 0.1448429095629314, 0.31...",0.000041,1.483332e-01,"(149, 2)",149,0.101208,HL73-650rev,original,False
2,m26,"[[1.0, 0.0024], [0.9995302195914396, 0.0024393...","[-0.11096748299540282, 0.19927498155118092, -0...","[0.12380172697643255, 0.3810484723612435, 0.18...",0.004796,1.952657e-01,"(149, 2)",149,0.133800,m26,original,False
3,p51dtip,"[[1.0, 0.0], [0.9995437754478239, 4.0810585862...","[-0.10960121031211903, -0.08457750965670288, -...","[0.1073547870445286, 0.10585432188376641, 0.26...",0.000000,7.965088e-02,"(149, 2)",149,0.114438,p51dtip,original,False
4,naca001064,"[[1.0, 0.001], [0.9995489620166345, 0.00106922...","[-0.13899473721335126, -0.09583862033946164, -...","[0.13899473721335134, 0.09583862033946101, 0.1...",0.002013,1.302785e-17,"(149, 2)",149,0.099972,naca001064,original,False


,airfoil_name,reason


## Optional Augmentation

Set `ENABLE_AUGMENTATION = True` to create smooth distorted copies of the base airfoils. The distortion is designed to be zero at the leading and trailing edges, which helps preserve the airfoil closure.

In [4]:
ENABLE_AUGMENTATION = True
AUGMENTATION_COPIES_PER_AIRFOIL = 2
AUGMENTATION_SEED = 42
MAX_CAMBER_DELTA = 0.010
MAX_THICKNESS_SCALE_DELTA = 0.080
MAX_LOCAL_BUMP = 0.006

def distort_coordinates(coords, rng):
    coords = np.asarray(coords, dtype=float)
    x = coords[:, 0]
    y = coords[:, 1].copy()

    envelope = np.sin(np.pi * x) ** 2
    sinusoid = np.sin(
        2 * np.pi * rng.uniform(0.5, 1.5) * x + rng.uniform(0.0, 2 * np.pi)
    )
    camber_shift = rng.uniform(-MAX_CAMBER_DELTA, MAX_CAMBER_DELTA)
    thickness_scale = 1.0 + rng.uniform(
        -MAX_THICKNESS_SCALE_DELTA,
        MAX_THICKNESS_SCALE_DELTA,
    )
    local_bump = rng.uniform(-MAX_LOCAL_BUMP, MAX_LOCAL_BUMP) * envelope * sinusoid

    y = y + (camber_shift * envelope) + local_bump
    y = y * (1.0 + (thickness_scale - 1.0) * envelope)

    augmented = asb.Airfoil(
        name="augmented",
        coordinates=np.column_stack([x, y]),
    ).normalize().repanel(N_POINTS_PER_SIDE)
    return augmented.coordinates

augmented_records = []
augmented_airfoils_dataset = pd.DataFrame()
augmentation_rejections = pd.DataFrame(columns=["airfoil_name", "reason"])

if ENABLE_AUGMENTATION:
    rng = np.random.default_rng(AUGMENTATION_SEED)
    for _, row in tqdm(base_airfoils_dataset.iterrows(), total=len(base_airfoils_dataset), desc="Augmenting airfoils"):
        for copy_idx in range(AUGMENTATION_COPIES_PER_AIRFOIL):
            coords = distort_coordinates(row["coordinates"], rng)
            augmented_records.append(
                {
                    "airfoil": asb.Airfoil(
                        name=f"{row['airfoil_name']}_aug_{copy_idx:02d}",
                        coordinates=coords,
                    ),
                    "source_airfoil_name": row["source_airfoil_name"],
                    "augmentation_tag": f"aug_{copy_idx:02d}",
                    "is_augmented": True,
                }
            )

    augmented_airfoils_dataset, augmentation_rejections = build_airfoil_records(augmented_records)

if ENABLE_AUGMENTATION and len(augmented_airfoils_dataset) > 0:
    airfoils_dataset = pd.concat([base_airfoils_dataset, augmented_airfoils_dataset], ignore_index=True)
else:
    airfoils_dataset = base_airfoils_dataset.copy()

airfoils_dataset = airfoils_dataset.reset_index(drop=True)
airfoils_dataset["airfoil_id"] = airfoils_dataset.index.map(lambda idx: f"airfoil_{idx:06d}")

print(f"Airfoils available for FiLMVAE: {len(airfoils_dataset)}")
print(f"Augmentation enabled: {ENABLE_AUGMENTATION}")
display(airfoils_dataset[["airfoil_id", "airfoil_name", "source_airfoil_name", "augmentation_tag", "is_augmented"]].head())
display(augmentation_rejections.head())

Building airfoil records: 100%|██████████| 4340/4340 [00:02<00:00, 2137.27it/s]

Airfoils available for FiLMVAE: 6510
Augmentation enabled: True


,airfoil_id,airfoil_name,source_airfoil_name,augmentation_tag,is_augmented
0,airfoil_000000,marske4,marske4,original,False
1,airfoil_000001,HL73-650rev,HL73-650rev,original,False
2,airfoil_000002,m26,m26,original,False
3,airfoil_000003,p51dtip,p51dtip,original,False
4,airfoil_000004,naca001064,naca001064,original,False


,airfoil_name,reason


## NeuralFoil Configuration

Each airfoil receives its own simulation configuration through `build_simulation_config`. You can change the logic there to run different Reynolds numbers, alpha ranges, transition assumptions, or model sizes for different airfoil families.

In [5]:
DEFAULT_SIMULATION_CONFIG = {
    "alpha_start": -6.0,
    "alpha_stop": 16.0,
    "alpha_step": 0.50,
    "Re": 1e6,
    "mach": 0.0, # Incompressible flow assumption
    "n_crit": 9.0,
    "xtr_upper": 1.0,
    "xtr_lower": 1.0,
    "model_size": "xlarge",
    "include_360_deg_effects": False,
}

def build_simulation_config(row):
    config = dict(DEFAULT_SIMULATION_CONFIG)

    # if row["thickness_proxy"] < 0.08:
    #     config["alpha_start"] = -4.0
    #     config["alpha_stop"] = 14.0

    # if row["is_augmented"]:
    #     config["Re"] = 8e5

    return config

airfoils_dataset["simulation_config"] = airfoils_dataset.apply(build_simulation_config, axis=1)
airfoils_dataset[["airfoil_name", "thickness_proxy", "simulation_config"]].head()

,airfoil_name,thickness_proxy,simulation_config
0,marske4,0.111491,"{'alpha_start': -6.0, 'alpha_stop': 16.0, 'alp..."
1,HL73-650rev,0.101208,"{'alpha_start': -6.0, 'alpha_stop': 16.0, 'alp..."
2,m26,0.133800,"{'alpha_start': -6.0, 'alpha_stop': 16.0, 'alp..."
3,p51dtip,0.114438,"{'alpha_start': -6.0, 'alpha_stop': 16.0, 'alp..."
4,naca001064,0.099972,"{'alpha_start': -6.0, 'alpha_stop': 16.0, 'alp..."


## Train/Validation Split

The split is performed before the aerodynamic expansion and grouped by `source_airfoil_name`, so original airfoils and their augmented variants stay in the same partition. This avoids leakage between train and validation.

In [6]:
VALIDATION_FRACTION = 0.10

unique_groups = sorted(airfoils_dataset["source_airfoil_name"].unique())
rng = np.random.default_rng(SEED)
n_val_groups = max(1, int(round(len(unique_groups) * VALIDATION_FRACTION)))
val_groups = set(rng.choice(unique_groups, size=n_val_groups, replace=False).tolist())

train_airfoils = airfoils_dataset[~airfoils_dataset["source_airfoil_name"].isin(val_groups)].reset_index(drop=True)
val_airfoils = airfoils_dataset[airfoils_dataset["source_airfoil_name"].isin(val_groups)].reset_index(drop=True)

print(f"Unique source profiles: {len(unique_groups)}")
print(f"Train grouped profiles: {len(train_airfoils)}")
print(f"Val grouped profiles:   {len(val_airfoils)}")

Unique source profiles: 2170
Train grouped profiles: 5859
Val grouped profiles:   651


## NeuralFoil Expansion to `(Cl, alpha)` Samples

In [7]:
def make_alpha_grid(config):
    alpha_values = np.arange(
        config["alpha_start"],
        config["alpha_stop"] + (0.5 * config["alpha_step"]),
        config["alpha_step"],
        dtype=float,
    )
    return alpha_values

def run_neuralfoil_for_airfoil(row):
    config = dict(row["simulation_config"])
    alpha_values = make_alpha_grid(config)

    airfoil = asb.Airfoil(
        name=row["airfoil_name"],
        coordinates=np.asarray(row["coordinates"], dtype=float),
    )

    repeated = lambda value: np.full_like(alpha_values, value, dtype=float)
    aero = airfoil.get_aero_from_neuralfoil(
        alpha=alpha_values,
        Re=repeated(config["Re"]),
        mach=repeated(config["mach"]),
        n_crit=repeated(config["n_crit"]),
        xtr_upper=repeated(config["xtr_upper"]),
        xtr_lower=repeated(config["xtr_lower"]),
        model_size=config["model_size"],
        include_360_deg_effects=config["include_360_deg_effects"],
    )

    cl = np.asarray(aero["CL"], dtype=float)
    cd = np.asarray(aero["CD"], dtype=float)
    cm = np.asarray(aero["CM"], dtype=float)

    valid_mask = np.isfinite(alpha_values) & np.isfinite(cl) & np.isfinite(cd) & np.isfinite(cm)
    polar_rows = []
    for idx in np.where(valid_mask)[0]:
        polar_rows.append(
            {
                "airfoil_id": row["airfoil_id"],
                "profile_id": row["airfoil_id"],
                "airfoil_name": row["airfoil_name"],
                "source_airfoil_name": row["source_airfoil_name"],
                "augmentation_tag": row["augmentation_tag"],
                "is_augmented": bool(row["is_augmented"]),
                "coordinates": np.asarray(row["coordinates"], dtype=float),
                "lower_weights": row["lower_weights"],
                "upper_weights": row["upper_weights"],
                "TE_thickness": float(row["TE_thickness"]),
                "leading_edge_weight": float(row["leading_edge_weight"]),
                "shape": row["shape"],
                "points": int(row["points"]),
                "alpha": float(alpha_values[idx]),
                "Cl": float(cl[idx]),
                "Cd": float(cd[idx]),
                "Cm": float(cm[idx]),
                "Re": float(config["Re"]),
                "mach": float(config["mach"]),
                "n_crit": float(config["n_crit"]),
                "xtr_upper": float(config["xtr_upper"]),
                "xtr_lower": float(config["xtr_lower"]),
                "model_size": config["model_size"],
            }
        )

    return polar_rows

def expand_airfoils_with_neuralfoil(airfoils_df, split_name):
    expanded_rows = []
    failed_airfoils = []

    iterator = tqdm(airfoils_df.iterrows(), total=len(airfoils_df), desc=f"NeuralFoil {split_name}")
    for _, row in iterator:
        try:
            expanded_rows.extend(run_neuralfoil_for_airfoil(row))
        except Exception as exc:
            failed_airfoils.append((row["airfoil_name"], f"{type(exc).__name__}: {exc}"))

    expanded_df = pd.DataFrame(expanded_rows)
    failed_df = pd.DataFrame(failed_airfoils, columns=["airfoil_name", "reason"])
    return expanded_df, failed_df

In [8]:
train_filmvae_dataset, train_nf_failures = expand_airfoils_with_neuralfoil(train_airfoils, "train")
val_filmvae_dataset, val_nf_failures = expand_airfoils_with_neuralfoil(val_airfoils, "validation")

print(f"Train FiLMVAE samples: {len(train_filmvae_dataset)}")
print(f"Val FiLMVAE samples:   {len(val_filmvae_dataset)}")
display(train_filmvae_dataset.head())
display(train_nf_failures.head())
display(val_nf_failures.head())

NeuralFoil validation: 100%|██████████| 651/651 [00:01<00:00, 394.17it/s]


Train FiLMVAE samples: 263655
Val FiLMVAE samples:   29295


,airfoil_id,profile_id,airfoil_name,source_airfoil_name,augmentation_tag,is_augmented,coordinates,lower_weights,upper_weights,TE_thickness,...,alpha,Cl,Cd,Cm,Re,mach,n_crit,xtr_upper,xtr_lower,model_size
0,airfoil_000000,airfoil_000000,marske4,marske4,original,False,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,...,-6.0,-0.476246,0.041542,-0.001280,1000000.0,0.0,9.0,1.0,1.0,xlarge
1,airfoil_000000,airfoil_000000,marske4,marske4,original,False,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,...,-5.5,-0.438151,0.034749,0.003855,1000000.0,0.0,9.0,1.0,1.0,xlarge
2,airfoil_000000,airfoil_000000,marske4,marske4,original,False,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,...,-5.0,-0.399521,0.029134,0.008135,1000000.0,0.0,9.0,1.0,1.0,xlarge
3,airfoil_000000,airfoil_000000,marske4,marske4,original,False,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,...,-4.5,-0.359610,0.023379,0.011642,1000000.0,0.0,9.0,1.0,1.0,xlarge
4,airfoil_000000,airfoil_000000,marske4,marske4,original,False,"[[1.0, 0.0], [0.9995355647272107, 4.8928458133...","[-0.05399972100481228, 0.007851682321366585, -...","[0.29987660240597813, 0.3079590445994013, 0.09...",0.000144,...,-4.0,-0.318235,0.017858,0.014320,1000000.0,0.0,9.0,1.0,1.0,xlarge


,airfoil_name,reason


,airfoil_name,reason


## Diagnostics

In [9]:
display(train_filmvae_dataset[["alpha", "Cl", "Cd", "Cm"]].describe())

diagnostic_sample = train_filmvae_dataset.sample(
    min(len(train_filmvae_dataset), 3000),
    random_state=SEED,
).copy()
diagnostic_sample["L_over_D"] = diagnostic_sample["Cl"] / diagnostic_sample["Cd"].clip(lower=1e-6)

fig = px.scatter(
    diagnostic_sample,
    x="alpha",
    y="Cl",
    color="source_airfoil_name",
    title="NeuralFoil Cl x alpha samples (train subset)",
)
fig.show()

fig = px.density_heatmap(
    diagnostic_sample,
    x="alpha",
    y="Cl",
    nbinsx=40,
    nbinsy=40,
    title="Sample density in the Cl-alpha plane",
    color_continuous_scale="Viridis",
)
fig.show()

fig = px.scatter(
    diagnostic_sample,
    x="Cl",
    y="Cd",
    color="alpha",
    title="Drag polar colored by alpha",
    color_continuous_scale="Turbo",
)
fig.show()

fig = px.histogram(
    diagnostic_sample,
    x="L_over_D",
    nbins=60,
    title="Lift-to-drag ratio distribution",
)
fig.show()

sample_count_per_airfoil = (
    train_filmvae_dataset.groupby("source_airfoil_name")
    .size()
    .reset_index(name="sample_count")
    .sort_values("sample_count", ascending=False)
    .head(30)
)

fig = px.bar(
    sample_count_per_airfoil,
    x="source_airfoil_name",
    y="sample_count",
    title="Top 30 airfoils by generated sample count",
)
fig.update_layout(xaxis_title="Source airfoil", yaxis_title="Generated samples")
fig.show()

fig = px.box(
    diagnostic_sample,
    x="is_augmented",
    y="Cl",
    color="is_augmented",
    title="Cl distribution for original vs augmented airfoils",
)
fig.show()

,alpha,Cl,Cd,Cm
count,263655.000000,263655.000000,263655.000000,263655.000000
mean,5.000000,0.764658,0.021332,-0.058358
std,6.493599,0.625514,0.028983,0.053722
min,-6.000000,-0.946948,0.003440,-0.424331
25%,-0.500000,0.256459,0.007701,-0.090076
50%,5.000000,0.845492,0.010661,-0.053548
75%,10.500000,1.285566,0.021684,-0.018979
max,16.000000,2.899522,0.239545,0.122858


In [10]:
display(train_filmvae_dataset.info())

<class 'pandas.DataFrame'>
RangeIndex: 263655 entries, 0 to 263654
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   airfoil_id           263655 non-null  str    
 1   profile_id           263655 non-null  str    
 2   airfoil_name         263655 non-null  str    
 3   source_airfoil_name  263655 non-null  str    
 4   augmentation_tag     263655 non-null  str    
 5   is_augmented         263655 non-null  bool   
 6   coordinates          263655 non-null  object 
 7   lower_weights        263655 non-null  object 
 8   upper_weights        263655 non-null  object 
 9   TE_thickness         263655 non-null  float64
 10  leading_edge_weight  263655 non-null  float64
 11  shape                263655 non-null  object 
 12  points               263655 non-null  int64  
 13  alpha                263655 non-null  float64
 14  Cl                   263655 non-null  float64
 15  Cd                   263655 

None

## Save FiLMVAE Datasets

The output files are saved with new names so the existing non-conditional Kulfan datasets are preserved.

In [11]:
train_output_path = processed_dir / f"train_filmvae_dataset_{N_WEIGHTS_PER_SIDE}.json"
val_output_path = processed_dir / f"val_filmvae_dataset_{N_WEIGHTS_PER_SIDE}.json"
metadata_output_path = processed_dir / f"filmvae_dataset_metadata_{N_WEIGHTS_PER_SIDE}.json"

train_filmvae_dataset.to_json(train_output_path)
val_filmvae_dataset.to_json(val_output_path)

metadata = {
    "n_weights_per_side": N_WEIGHTS_PER_SIDE,
    "n_points_per_side": N_POINTS_PER_SIDE,
    "train_samples": int(len(train_filmvae_dataset)),
    "val_samples": int(len(val_filmvae_dataset)),
    "augmentation_enabled": bool(ENABLE_AUGMENTATION),
    "augmentation_copies_per_airfoil": int(AUGMENTATION_COPIES_PER_AIRFOIL),
    "default_simulation_config": DEFAULT_SIMULATION_CONFIG,
    "condition_columns": ["Cl", "alpha"],
}
metadata_output_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved train dataset: {train_output_path}")
print(f"Saved val dataset:   {val_output_path}")
print(f"Saved metadata:      {metadata_output_path}")

Saved train dataset: /home/matsouto/Documents/py/AeroGen/data/processed/train_filmvae_dataset_8.json
Saved val dataset:   /home/matsouto/Documents/py/AeroGen/data/processed/val_filmvae_dataset_8.json
Saved metadata:      /home/matsouto/Documents/py/AeroGen/data/processed/filmvae_dataset_metadata_8.json
